# SANS analysis II: Priors and statistics 

You have now [simulated](./../3-mcstas/mcstas-sans.ipynb), [reduced](./../4-reduction/reduction-sans.ipynb), and carried out [routine analysis of your SANS data](analysis-sans.ipynb).

This notebook builds on the previous by beginning to look at model fitting [probabilistically](./prob_data.ipynb), including prior knowledge about the system or parameters under study in the analysis. 

We will continue with the same dataset. 

In [ ]:
import numpy as np
import utils
import matplotlib.pyplot as plt
from sans_fitter import SANSFitter



As before, we can load in our simulated (or example) dataset and check it looks correct:

In [ ]:
filename = utils.fetch_data("4-reduction/sans_iofq.dat")
#filename = '../4-reduction/sans_iofq.dat'


q, i, di = utils.load(filename)
fig, ax = plt.subplots()
ax.errorbar(q, i, di, fmt='.')
ax.set(yscale='log', xscale='log', xlabel='$q$/Å^-1', ylabel='I(q)')
plt.show()

Now we can load in sans-fitter and initialise it with the same sphere model as before:

In [ ]:
fitter = SANSFitter()
fitter.load_data(filename)
fitter.set_model('sphere')
fitter.get_params()

By default, some parameters are set but nothing is allowed to vary. Obviously, this will constrain our fitting algorithm excessively. 

In the previous example, we have included prior knowledge in the analysis through the use of bounded parameters. 

Bounded parameters cannot have values less than some lower bound (Min) or greater than some upper bound (Max), as the probability of the parameters having these values is zero. 
For example, if the parameter `b` from a quadratic model has bounds of 0 and 10, then there is an equal probability that the value of `b` can be anything in between 0 and 10, and a probability of 0 outside those bounds, i.e., it has a uniform prior probability distribution ({numref}`uniform`).

## Exercise 1: Provide some sensible bounds for the data, and confirm the input:

In [ ]:
fitter.set_param('sld', value=3, min=1, max=30, vary=False)
fitter.set_param('sld_solvent', value=6, min=1, max=30, vary=False)

fitter.set_param('radius', value=80, min=10, max=300, vary=True)
fitter.set_param('scale', value=1.4e-7, min=0, max=1, vary=True)
fitter.set_param('background', value=0.1, min=0, max=1, vary=True)

fitter.get_params()

Great: Now the model is set up, let's introduce some new syntax for $Bayesian$ analysis using `sans-fitter`. 

`SasView`, and hence `sans-fitter`, have a number of optimisation algorithms built-in. One of these is `DREAM`, a population based algorithm. 

`DREAM` is relatively slow; it follows a differential evolution-like process but sometimes keeps individuals which get worse with the evolution and allows these to progress as a Markov chain which converges on the equilibrium distribution, where the chain draws randomly from the posterior distribution. 

Therefore, we can use the `DREAM` fitting algorithm to determine parameter uncertainties from our fitting process. 

[More can be read about `DREAM` in the associated publication](https://www.degruyterbrill.com/document/doi/10.1515/IJNSNS.2009.10.3.273/html), or in the [SasView documentation](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/optimizer.html#fit-dream). 

We will begin by setting a few parameters: `samples` (number of points to be drawn from the Markov chain) and `burn` (number of iterations for the Markov chain to converge to the equilibrium distribution).

To estimate the 68% interval to two digits of precision, at least 1e5 (or 100,000) samples are needed. For the 95% interval, 1e6 (or 1,000,000) samples are needed. 1e4 samples gives a 'quick-and-dirty' approximation of the uncertainty. 

In [ ]:
result = fitter.fit_bayesian(samples=10000, burn=100)

We can plot the fit and residuals as before: 

In [ ]:
fitter.plot_results(show_residuals=True, log_scale=True)

We can now look at the fitting output in detail using a variety of plots:


i. A 'corner' plot showing a grid of parameter distribution from the Bayesian multi-parameter analysis.

ii. A marginal posterior plot showing the probability distribution for a single, chosen parameter.

iii. A Bayesian posterior predictive 95% credible band plot, which displays the model's predicted outcomes over a range of inputs (shaded region may be invisible depending on constraints).

iv. A parameter heatmap giving a colour-coded grid of relationships and statistical dependencies between model parameters estimated from Bayesian inference.

v. The Markov chain Monte Carlo (MCMC) trace which demonstrates how all of the parameters evolved during the fit.

In [ ]:
print('\nGenerating posterior pair (corner) plot...')
fitter.plot_posterior_pairs()

In [ ]:
print('\nGenerating marginal posterior for radius...')
fitter.plot_param_distribution('radius')

In [ ]:
print('\nGenerating posterior predictive band...')
fitter.plot_posterior_predictive(style='band')

In [ ]:
print('\nGenerating parameter correlation heatmap...')
fitter.plot_param_correlations()

In [ ]:
print('\nGenerating MCMC trace plot...')
fitter.plot_trace()

Finally, an example of how one can access the posterior data and save the fit results:

In [ ]:
posterior = fitter.get_posterior()
print('\nSampled parameters:', posterior.labels)
print('Chain shape:', posterior.samples.shape)
print('95% credible intervals:')
for name in posterior.labels:
    low, high = posterior.ci_95[name]
    print(f'  {name}: [{low:.6g}, {high:.6g}]')

# Export the raw chain for external analysis (pandas, corner, arviz, ...)
posterior.save_posterior_csv('posterior_chain.csv')
print('\n✓ Raw posterior chain saved to posterior_chain.csv')

# The saved fit results include the credible intervals in the header
fitter.save_results('bayesian_fit_results.csv')


## Exercise 2: Explore models for the data

You are now armed with knowledge of how to set models, variable and fixed parameters and constraints, and to then fit data with straightforward and more statistically rigorous approaches.

In the previous notebook we explored this dataset using ellipsoidal and spherical fits. Can Bayesian analysis help you to distinguish between these?

Let's start by fetching all available models; documentation on these `SasView` models can be found [here](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/models/index.html). 

Pick one, or several, and see how they perform for this dataset!

Further work:
You could also explore the effects of limiting the fitted q-range to see how this limitation affects statistical distributions: $$$ fitter.set_q_range(qmin=0.1, qmax=0.3)

or 

Try adding a simple structure factor: $$$ fitter.set_structure_factor('hardsphere', radius_effective_mode='link_radius')

In [ ]:
from sans_fitter import get_all_models
print(get_all_models())

In [ ]:
from sans_fitter import SANSFitter
filename = utils.fetch_data("4-reduction/sans_iofq.dat")

fitteri = SANSFitter()
fitteri.load_data(filename)
fitteri.set_model('cylinder')
fitteri.get_params()

In [ ]:
fitteri.set_param('sld', value=3, min=1, max=30, vary=False)
fitteri.set_param('sld_solvent', value=6, min=1, max=30, vary=False)

fitteri.set_param('radius', value=80, min=10, max=300, vary=True)
fitteri.set_param('length', value=80, min=10, max=300, vary=True)

fitteri.set_param('scale', value=1.4e-7, min=0, max=1, vary=True)
fitteri.set_param('background', value=0.1, min=0, max=1, vary=True)

fitteri.get_params()

In [ ]:
result = fitteri.fit_bayesian(samples=10000, burn=100)
fitteri.plot_results(show_residuals=True, log_scale=True)

In [ ]:
print('\nGenerating posterior pair (corner) plot...')
fitteri.plot_posterior_pairs()

In [ ]:
from sans_fitter import SANSFitter
filename = utils.fetch_data("4-reduction/sans_iofq.dat")

fitteri = SANSFitter()
fitteri.load_data(filename)
fitteri.set_model('vesicle')
fitteri.get_params()

In [ ]:
fitteri.set_param('sld', value=3, min=1, max=30, vary=False)
fitteri.set_param('sld_solvent', value=6, min=1, max=30, vary=False)

fitteri.set_param('radius', value=80, min=10, max=300, vary=True)
fitteri.set_param('thickness', value=10, min=5, max=100, vary=True)
fitteri.set_param('volfraction', value=0.1, min=0, max=1, vary=True)


fitteri.set_param('scale', value=1, min=0, max=1, vary=False)
fitteri.set_param('background', value=0.1, min=0, max=1, vary=True)

fitteri.get_params()

In [ ]:
result = fitteri.fit_bayesian(samples=10000, burn=100)
fitteri.plot_results(show_residuals=True, log_scale=True)

In [ ]:
print('\nGenerating posterior pair (corner) plot...')
fitteri.plot_posterior_pairs()

In [ ]:
print('\nGenerating posterior predictive band...')
fitteri.plot_posterior_predictive(style='band')